# Active Recall

How do I best remember the key aspects of an implementation so that I can reproduce a piece of code from scratch?<br/>
Right now my thoughts are
- Try to remember the tests name and make sure the test name really connects with what were are trying to do. Here, we test ourselves by stubbing
  out all the test by memory writing in pseudo the steps involved in the test implementation. Ensure we relate everything back to the overall goal.
  A useful tactic here is having like guiding questions that help us tie everything together. What, Why, How, Where, When questions.
- Taking at page from how artists generally produce or reproduce a piece of art quickly
    1. Stub out a general outline of a function/method, struct, `impl` block or trait 
    2. Add initial general detail
    3. Add finer details last

Although it might look like we are trying to memorize the code word for word, they key idea here is how best to fill in the blanks when working on an implementation.<br/>
i.e. what are they key details that we all blanking out on, why, and why are the details important?

My assumption is that this will help in structuring our thinking more clearly and make our own implementation straightforwad because we have a framework in place that
helps us layout our implementations, from high level to low level details.

Honestly, the more I think of this the more it turns out that the leetcode style of solving problems is what we are using to think about project implementation.

# 7.0 Reject Invalid Subscriber #2

## 7.7. Sending A Confirmation Email

### 7.7.x Actix

#### 7.7.1. Static Email

##### 7.7.1.0. Oveview

_**Why?**_<br/>
Check that when we get a new subscriber they get an confirmation email.

_**How?**_<br/>
- First add a test that checks a new subscriber is sent an email. We'll use `wiremock::Mock` to mock the email server that mocks sending
  and email and returning a `200 OK` if it receives the send email request.
- Extract the email client into `subscribe` in order to call `email_client.send_emai()`

##### 7.7.1.1. Red Test

_**What?**_
- What is the name of the test?
  - expected - `subscribe_send_email_for_valid_form_data`
  - actual - `subscribe_sends_confirmation_email_for_valid_form_data`
  - diff - _`sends_confirmation`_
- Stub out solution from memory
```Rust
//! tests/api/helpers.rs ✅

use wiremock::MockServer;

use zero2prod::email_client::EmailClient;

pub struct TestApp {
    pub address: String,
    pub db_pool: PgPool,
    pub email_server: MockServer
}

pub async fn spawn_app() -> TestApp {
    // [...]
    let email_server = MockServer::start().await;
    let configuration = {
        let config = get_config().expect("Failed to read configuration files");
        // [...]
        config.email_client.base_url = &email_server.uri();
        config
    };
    TestApp {
        // [...]
        email_server
    }
}
```
```Rust
//! test/api/subscriptions.rs

#[tokio::test]
async fn subscribe_sends_confirmation_email_on_valid_form_data() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .expect(1) // -> Assertion will be done at tne end of the scope
        .mount(&app.email_server)
        .await; ⚠️

    // Act
    app.post_subscriptions(&body.into()).await; ⚠️

    // Assert
    // Mocke assert on drop ⚠️
}
```
- Update markdown with screenshot.

##### 7.7.1.2. Green Test

_**What?**_<br/>
- Capture `email_client` in `subscribe`
- Make `email_client.send_email` call with dummy data

**Expected Implementation**
```Rust
//! src/routes/subscriptions.rs

pub async fn subscribe(
    form: Form<FormData>, // ?? ❌
    db_pool: web::Data<PgPool>,
    email_client: web::Data<EmailClient>,
) -> HttpResponse {

    // [...]
    
    let new_subscriber = match FormData { // ?? ❌
        Ok(new_subscriber) => new_subscriber,
        Err(_) => return HttpResponse::InternalServerError, // ?? ❌
    };

    if insert_subscriber(&db_pool, &new_subscriber)
        .await
        .is_err()
    {
        return HttpResponse::InternalServerError().finish()
    }
    
    if email_client.send_email(
        &new_subscriber.email, // ?? ❌
        "Welcome",
        "Placeholder HtmlBody",
        "Placeholder TextBody",
    )
    .await
    .is_err()
    {
        return HttpResponse::InternalServerError().finish()
    }
    HttpResponse::Ok().finish()
    
}
```

**Actual Implementation: _Misses_**
```Rust
pub async fn subscribe(
    form: web::Form<FormData>,
    db_pool: web::Data<PgPool>,
    email_client: web::Data<PgPool>,
) {
    let new_subscriber = match form.0.try_into() {
        Ok(form) => form,
        Err(_) => HttpResponse::BadRequest().finish(),
    };

    // [...]
    
    if email_client.send_email(
        new_subscriber.email, // Notice how this call consumes `new_subscriber`
        "Welcome",
        "Placeholder HtmlBody",
        "Placeholder TextBody",
    )
    .await
    // [...]
}
```

#### 7.7.2. A Static Confirmation Link

##### 7.7.2.0. Overview

**Why?** <br/>
We want to ensure that when a user is sent the confirmation email it has a confirmation link

**How?**
- We add a test that checks that the confirmation email that received by a new subscriber it has a confirmation link. For now we can use a dummy link.
- Update the `HtmlBody` and `TextBody` to include a a clickable link.

##### 7.7.2.1. Red Test

**What?**<br/>
- What is the name of the test?
  - _expected_: `subscribe_sends_confirmation_email_with_confirmation_link`
  - _actual_: `subscribe_sends_confirmation_email_with_a_link`
  - _diff_: `confirmation_link`
- What is the test source?<br/>
_Expected_
```Rust
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    // Wire up mock email server
    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;

    // Trigger confirmation email send
    app.post_subscriptions(body.into()).await;

    // Act
    let email_request = &app.email_server.received_request().unwrap(); // ?? ❌

    // Extract link from email
    let get_link = |s: &str| { // ?? ❌
        let link = linkify::FetchUrl().url()
            .filter()
            .collect();
        assert_eq!(link.len(), 1);
    };

    let html_body = get_links(&email_request["HtmlBody"]).unwrap()[0]; // ?? ❌
    let text_body = get_links(&email_request["TextBody"]).unwrap()[0]; // ?? ❌
    
    // Assert
    assert_eq!(html_body, text_body);
}
```
_Actual_
```Rust
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // Arrange
    let app = spawn_app().await;
    let body = "username=lei%20yin&email=lei_yin_loo%40gmail.com";

    // wire up mock email server
    Mock::given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;

    // Act
    // trigger email send
    app.post_subscriptions(body.into()).await;
    
    // Assert
    
    // Get the first intercepted request
    let email_request = &app.email_server.received_requests().await.unwrap()[0];
    // Parse the body as JSON, starting from raw bytes
    let body: serde_json::Value = serde_json::from_slice(&email_request.body)
        .unwrap();

    // closure to extract link from string
    let get_links = |s: &str| {
        let links: Vec<_> = linkify::LinkFinder::new()
            .links(s)
            .filter(|l| *l.kind() == linkify::LinkKind::Url)
            .collect();
        assert_eq!(links.len(), 1);
        links[0].as_str().to_owned()
    };

    let html_body = get_links(&body["HtmlBody"].as_str().unwrap());
    let text_body = get_links(&body["TextBody"].as_str().unwrap());

    assert_eq!(html_body, text_body);
}
```

_Diff_
```Rust
#[tokio::test]
async fn subscribe_sends_confirmation_email_with_confirmation_link() {
    // [...]

    // Assert
    // Intercepting first request to email_server
    let email_request = &app.email_server.received_requests().await.unrap()[0];
    // Parse the body as JSON, starting from raw bytes.
    let body: serde_json::Value = serde_json::from_slice(&email_request.body).unwrap();

    let get_links = |s: &str| {
        let links: Vec<_> = linkify::LinkFinder::new()
            .links(s)
            .filter(| l | *l.king() == linkify::LinkKing::Url )
            .collect();
        assert_eq!(links.len(), 1);
        links[0].as_str().to_owned()
    };

    let html_body = get_links(&body["HtmlBody"].as_str().unwrap());
    let text_body = get_links(&body["TextBody"].as_str().unwrap());

    assert_eq!(html_body, text_body);
}
```

##### 7.7.2.2. Green Test

**What?**
- Add a dummy `confirmation_link` to `email_client.send_email()`

_Expected_
```Rust
pub async fn subscribe(/* */) -> HttpResponse {
    //  [...]
    let confirmation_link = "https://dummy-placeholder-domain.com/subscriptions/confirm";
    if email_client
        .send_email(
        new_subscriber.email
        "Welcome!"
        format!(
            "Welcome to our newsletter!<br />\
            Click <a href=\"{}\">here</a> to confirm your subscription.",
            confirmation_link
        ),
        format!(
           "Welcome to our newsletter!\nVisit {} to confirm your subscription.",
            confirmation_link,
        ),
    )
    .await
    .is_err()
    {
        return HttpResponse::InternalServerError().finish();
    }
}
```
_Actual_: ✅

_Diff_: 👌

##### 7.7.2.3. Refactor

**What?**<br/>
- extract out `email_client.send_email` into its seprate function that we call in `subscribe`

**Why?**<br/>
- `subscribe` is being polluted by `email_client.send_email` call. We extract it out the call reason about what `subscribe` does more clearly.

#### 7.7.3. Pending Confrimation

##### 7.7.3.0. Overiview

Here, in the book we start by first refactoring the test `subscribe_returns_200_for_valid_form_data` and extracting out the
persistency checks into it own test `subscribe_persists_new_subscriber`. After this we go on to update the new test to assert
that the default `status` for a new subscriber is `pending_confirmation`. 

How we'll do this is a bit different. We will do the refactoring and extracting the persistency checks at the end, following the 
_Red, Green, Refactor_ steps a bit more strictly. This is not necessary. Just doing it this way in an attempt to drill in the TDD
cycle.

_**The Wny?**_<br/>
Here we want to ensure that a new subscriber's `status` is set to `pending_confirmation` to allow us to send a confirmation email
with at confirmation link. When the new subscriber clicks on the confirmaation link that when we update their `status` to confirmed.

_**The How?**_<br/>
Right now the default `status` is `confirmed`. We need to change that.

##### 7.7.3.1. Red Test

_**The What?**_<br/>

We add a test to ensure that the initial `status` of a new subscriber is `pending_confirmation` when they are first inserted into our db.<br/>
For now we will just add an assertion to `subscribe_returns_200_for_valid_form_data` to check the appropriate status for our _Red step._

_Expected_
```Rust
async fn subscribe_returns_200_for_valid_form_data() {
    // [...]
    // We update the query.
    let saved = sqlx.query!("SELECT username, email, status FROM subscriptions",) // ❌
        .fetch_one(&db_pool) // ❌
        .await
        .expect("Failed to execute db query");  // ❌

    assert_eq!(saved.username, "lei yin");
    assert_eq!(saved.email, "lei_yin_loo@gmail.com");
    assert_eq!(saved.status, "pending_confirmation");
}
```

_Actual_
```Rust
async fn subscribe_returns_200_for_valid_form_data(){
    // [...]

    // Update our query with status
    let saved = sqlx::query!("SELECT email, username, status FROM subscriptions")
        .fetch_one(&app.db_pool)
        .await
        .expect("Failed to fetch saved subscription in test");

    assert_eq!(saved.email, "lei_yin_loo@gmail.com");
    assert_eq!(saved.username, "lei yin");
    assert_eq!(saved.status, "pending_confirmation");
}
```

_Diff_
```Rust
let saved = sqlx::query!("SELECT email, username, status FROM subscriptions", )
    .fetch_one(&app.db_pool)
    .await
    .expect("Failed to fetch saved subscription in test");
```

##### 7.7.3.2. Green Test

_**The What**_<br/>
Here it is very simple. We update the `insert_subscribe` function default from the initial `confirm` to `pending_confirmation`.

_Expected_
```Rust
pub async fn insert_subscriber(db_pool: PgPool, new_sbuscriber: &NewSubscriber) -> Result<(), sqlx::Error> { // ❌
    // [...]
    sqlx.query!(
        r#"
            INSERT INTO subscriptions (id, email, username, subscribed_at, status)
            VALUES ($1, $2, $3, $4, 'pending_confirmation')
        "#,
        Uuid::now_v7(),
        new_subscriber.email, // ❌
        mew_subscriber.username, // ❌
        Utc::now(),
    )
    .execute(&db_pool) // ❌
    .await
    .map_err(|e| {
        // [...]
    })?;

    Ok(())
}
```

_Actual_
```Rust
pub async fn insert_subscriber(db_pool: &PgPool, new_sbuscriber: &NewSubscriber) -> Result<(), sqlx::Error> {
    sqlx.query!(
        r#"
            INSERT INTO subscriptions (id, email, username, subscribed_at, status)
            VALUES ($1, $2, $3, $4, 'pending_confirmation')
        "#,
        Uuid::now_v7(),
        new_subscriber.email.as_ref(),
        mew_subscriber.username.as_ref(),
        Utc::now(),
    )
    .execute(db_pool)
    .await
    .map_err(|e| {
        // [...]
    })?;

    Ok(())
}
```

_Diff_
```Rust
pub async fn insert_subscriber(db_pool: &PgPool, new_sbuscriber: &NewSubscriber) -> Result<(), sqlx::Error> {
    sqlx.query!(
        // [...]
        new_subscriber.email.as_ref(),
        new_subscriber.username.as_ref(),
        Utc::now(),
    )
    .execute(&db_pool)
    .await
    // [...]

    Ok(())
}
```

##### 7.7.3.3. Refactor

_**What?**_<br/>

- Seperate out the persistency assertion from `subscribe_returns_200_for_valid_form_data` into separete `subscribe_persists_new_subscriber`

_**Why?**_<br/>
We want the intention for each test to be clear. On tests the response status the other checks that the data was persisted correctly.

_**How?**_

_Expected_
```Rust
#[tokio::test]
async fn subscribe_returns_200_for_valid_form_data() {
    // [...]
    // Assert
    assert_eq!(response.status().as_u16(), 200);

    // Removes the rest
}

#[tokio::test]
async fn subscribe_persists_new_subscriber() {
    let app = spawn_app().await;
    let body = "usernamelei%20yin&email=lei_yin_loo%40.gmail.com";

    Mock.given(path("/email"))
        .and(method("POST"))
        .respond_with(ResponseTemplate::new(200))
        .mount(&app.email_server)
        .await;

    app.post_subscriptions(body.into()).await;

    let saved = sqlx::query!("SELECT email, username, status FROM subscriptions")
        .fetch_one(&app.db_pool)
        .await
        .except("Failed to fetch saved subscriber in test");

    assert_eq!(saved.email, "lei_yin_loo@gmail.com");
    assert_eq!(saved.username, "lei yin");
    assert_eq!(saved.status, "pending_confirmation");
}
```

_Actual_: ✅
_Diff_: 👌

### 7.7.x Axum